# Relevant Python 3.12 Changes — Advanced Tutorial Problems with Solutions

Python 3.12 introduced several language, typing, standard-library, performance, and tooling changes.

This notebook does **not** attempt to list every Python 3.12 change.

Instead, we will focus on a selection of changes that are especially useful when writing:

- reusable libraries;
- data-processing pipelines;
- typed application code;
- filesystem tools;
- numerical code;
- debuggers, profilers, and coverage tools.

The notebook follows a tutorial style:

1. introduce the feature;
2. begin with a small example;
3. identify a realistic problem;
4. break the problem into logical steps;
5. build the solution gradually;
6. test normal cases and edge cases;
7. discuss design decisions and common mistakes.

For the complete release notes, see:

- [What's New in Python 3.12](https://docs.python.org/3/whatsnew/3.12.html)
- [Python 3.12 documentation](https://docs.python.org/3.12/)

## Topics covered

We will build advanced examples around:

1. PEP 701 — improved f-string syntax;
2. PEP 695 — new generic and type-alias syntax;
3. PEP 692 — precise typing for `**kwargs`;
4. PEP 698 — the `@override` decorator;
5. PEP 709 — inlined comprehensions;
6. `itertools.batched`;
7. `pathlib.Path.walk`;
8. `math.sumprod` and the new `steps` argument for `math.nextafter`;
9. PEP 688 — implementing the buffer protocol in Python;
10. PEP 669 — low-impact monitoring with `sys.monitoring`;
11. a capstone problem combining several Python 3.12 features.

## Runtime requirement

This notebook intentionally contains Python 3.12-only syntax.

For example:

```python
def first[T](items: list[T]) -> T:
    ...
```

A Python 3.11 interpreter cannot even parse that syntax.

We therefore begin by checking the interpreter version.

In [1]:
from __future__ import annotations

import ast
import dis
import hashlib
import inspect
import math
import shutil
import sys
import tempfile
import timeit

from collections import Counter, defaultdict, deque
from collections.abc import Buffer, Callable, Hashable, Iterable, Iterator, Mapping, Sequence
from dataclasses import dataclass
from itertools import batched
from pathlib import Path
from typing import NotRequired, Required, TypedDict, Unpack, override

assert sys.version_info >= (3, 12), (
    "This notebook requires Python 3.12 or newer. "
    f"Current version: {sys.version.split()[0]}"
)

print("Running on Python", sys.version.split()[0])

Running on Python 3.13.7


## Small assertion helpers

The tutorial uses plain assertions so that every solution remains easy to copy into a normal Python file.

The following helpers provide clearer error messages.

In [2]:
def check_equal(actual: object, expected: object, *, label: str = "") -> None:
    assert actual == expected, (
        f"{label + ': ' if label else ''}"
        f"expected {expected!r}, got {actual!r}"
    )


def check_close(
    actual: float,
    expected: float,
    *,
    rel_tol: float = 1e-12,
    abs_tol: float = 0.0,
    label: str = "",
) -> None:
    assert math.isclose(actual, expected, rel_tol=rel_tol, abs_tol=abs_tol), (
        f"{label + ': ' if label else ''}"
        f"expected approximately {expected!r}, got {actual!r}"
    )


def check_raises(
    exception_type: type[BaseException],
    func: Callable[..., object],
    /,
    *args: object,
    contains: str | None = None,
    **kwargs: object,
) -> BaseException:
    try:
        func(*args, **kwargs)
    except exception_type as exc:
        if contains is not None:
            assert contains in str(exc), (
                f"expected {contains!r} in exception message {str(exc)!r}"
            )
        return exc
    except BaseException as exc:
        raise AssertionError(
            f"expected {exception_type.__name__}, got {type(exc).__name__}"
        ) from exc

    raise AssertionError(f"expected {exception_type.__name__} to be raised")

# 1. PEP 701 — More flexible f-strings

F-strings existed long before Python 3.12.

However, the older implementation imposed several grammar restrictions.

Python 3.12 formally integrates f-strings into Python's grammar. Among other improvements, we can now:

- reuse the same quote character inside a replacement expression;
- write multi-line replacement expressions more naturally;
- place comments inside replacement fields;
- nest f-strings more consistently;
- receive better syntax errors.

## A small quote-reuse example

Before Python 3.12, developers often changed the outer quote character merely to avoid conflicts.

In Python 3.12, this is valid:

In [3]:
user = {
    "name": "Grace",
    "role": "compiler engineer",
}

description = f"{user["name"]} works as a {user["role"]}."
print(description)

Grace works as a compiler engineer.


The outer f-string and the dictionary keys both use double quotes.

This may look like a cosmetic improvement, but it becomes useful when rendering deeply nested structured data.

## Multi-line expressions and comments

A replacement field may now contain a multi-line expression and comments.

This can improve readability when the expression is genuinely complex.

It should not be used as an excuse to place an entire program inside an f-string.

In [4]:
measurements = [3.5, 4.25, 5.0, 6.75]

summary = f"""count={
    len(measurements)
}
mean={
    math.fsum(
        measurements  # fsum gives a numerically careful floating-point sum.
    ) / len(measurements)
:.2f}"""

print(summary)

count=4
mean=4.88


## Advanced Problem 1 — Build a benchmark table formatter

Suppose a benchmark runner produces records such as:

```python
{
    "name": "parse-large-json",
    "seconds": 0.04137,
    "iterations": 250,
    "baseline": 0.05210,
}
```

We want a compact table with:

- a dynamically sized name column;
- a dynamically selected numeric precision;
- a percentage change from the baseline;
- a marker for regressions;
- a final summary line.

A positive percentage means the benchmark became slower.

A negative percentage means it became faster.

### Desired output shape

The exact column widths depend on the input.

A result might look like:

```text
benchmark             seconds     iterations      change
parse-large-json       0.0414            250     -20.60%
sort-records           0.0831            100       5.19% !
----------------------------------------------------------------
mean                    0.0623
```

The exclamation mark marks a regression whose percentage exceeds a configurable threshold.

### Step 1 — Calculate percentage change

The standard percentage-change formula is:

```text
(current - baseline) / baseline * 100
```

We should reject a zero or negative baseline because it does not represent a meaningful timing baseline.

In [5]:
def percentage_change(current: float, baseline: float) -> float:
    if baseline <= 0:
        raise ValueError("baseline must be positive")
    return (current - baseline) / baseline * 100


check_close(percentage_change(8, 10), -20.0)
check_close(percentage_change(10.5, 10), 5.0)
check_raises(ValueError, percentage_change, 1, 0)

ValueError('baseline must be positive')

### Step 2 — Decide the dynamic widths

The name column must be wide enough for:

- the header `"benchmark"`;
- the longest benchmark name.

The remaining columns can use fixed minimum widths.

We will keep the width calculation separate from rendering. This makes the code easier to test.

In [6]:
class BenchmarkRow(TypedDict):
    name: str
    seconds: float
    iterations: int
    baseline: float


def benchmark_name_width(rows: Sequence[BenchmarkRow]) -> int:
    return max(
        len("benchmark"),
        *(len(row["name"]) for row in rows),
    )


sample_benchmarks: list[BenchmarkRow] = [
    {
        "name": "parse-large-json",
        "seconds": 0.04137,
        "iterations": 250,
        "baseline": 0.05210,
    },
    {
        "name": "sort-records",
        "seconds": 0.08310,
        "iterations": 100,
        "baseline": 0.07900,
    },
]

print(benchmark_name_width(sample_benchmarks))

16


### Step 3 — Render one row

The format specification itself may contain expressions.

For example:

```python
f"{value:>{width}.{precision}f}"
```

Here both the field width and precision are dynamic.

In [7]:
def render_benchmark_row(
    row: BenchmarkRow,
    *,
    name_width: int,
    precision: int,
    regression_threshold: float,
) -> str:
    change = percentage_change(row["seconds"], row["baseline"])
    marker = " !" if change > regression_threshold else ""

    return (
        f"{row["name"]:<{name_width}}"
        f"  {row["seconds"]:>10.{precision}f}"
        f"  {row["iterations"]:>12d}"
        f"  {change:>10.2f}%{marker}"
    )


print(
    render_benchmark_row(
        sample_benchmarks[0],
        name_width=benchmark_name_width(sample_benchmarks),
        precision=4,
        regression_threshold=3.0,
    )
)

parse-large-json      0.0414           250      -20.60%


### Step 4 — Build the complete table

Now we can combine:

- validation;
- dynamic name width;
- a header;
- rendered rows;
- a separator;
- a mean timing.

Notice that `math.fsum` is used for the average.

### Complete Solution 1

In [8]:
def render_benchmark_table(
    rows: Sequence[BenchmarkRow],
    *,
    precision: int = 4,
    regression_threshold: float = 3.0,
) -> str:
    if not rows:
        raise ValueError("at least one benchmark row is required")
    if precision < 0:
        raise ValueError("precision must be non-negative")

    for row in rows:
        if row["seconds"] < 0:
            raise ValueError("seconds must be non-negative")
        if row["iterations"] <= 0:
            raise ValueError("iterations must be positive")
        if row["baseline"] <= 0:
            raise ValueError("baseline must be positive")

    name_width = benchmark_name_width(rows)

    header = (
        f"{"benchmark":<{name_width}}"
        f"  {"seconds":>10}"
        f"  {"iterations":>12}"
        f"  {"change":>11}"
    )

    rendered_rows = [
        render_benchmark_row(
            row,
            name_width=name_width,
            precision=precision,
            regression_threshold=regression_threshold,
        )
        for row in rows
    ]

    mean_seconds = math.fsum(row["seconds"] for row in rows) / len(rows)
    separator = "-" * len(header)

    footer = (
        f"{"mean":<{name_width}}"
        f"  {mean_seconds:>10.{precision}f}"
    )

    return "\n".join([header, *rendered_rows, separator, footer])


benchmark_table = render_benchmark_table(
    sample_benchmarks,
    precision=4,
    regression_threshold=3.0,
)

print(benchmark_table)

benchmark            seconds    iterations       change
parse-large-json      0.0414           250      -20.60%
sort-records          0.0831           100        5.19% !
-------------------------------------------------------
mean                  0.0622


### Step 5 — Test the formatter

We should test more than the happy path.

Important edge cases include:

- empty input;
- invalid precision;
- invalid iteration count;
- zero baseline;
- a regression marker appearing only when appropriate.

In [9]:
assert "parse-large-json" in benchmark_table
assert "sort-records" in benchmark_table
assert "-20.60%" in benchmark_table
assert "5.19% !" in benchmark_table
assert "mean" in benchmark_table

check_raises(
    ValueError,
    render_benchmark_table,
    [],
    contains="at least one",
)
check_raises(
    ValueError,
    render_benchmark_table,
    sample_benchmarks,
    precision=-1,
)
check_raises(
    ValueError,
    render_benchmark_table,
    [
        {
            "name": "bad",
            "seconds": 1.0,
            "iterations": 0,
            "baseline": 1.0,
        }
    ],
)

ValueError('iterations must be positive')

### Discussion

The most important design decision was not the f-string syntax.

It was separating the problem into:

1. percentage calculation;
2. width calculation;
3. one-row rendering;
4. whole-table rendering;
5. validation and tests.

Python 3.12's f-string improvements make the final formatting code easier to read, but good decomposition still matters more than clever formatting.

# 2. PEP 695 — New generic syntax and type aliases

Before Python 3.12, generic code typically introduced `TypeVar` objects manually.

Python 3.12 provides dedicated syntax for type parameters.

For example:

In [10]:
def last[T](items: Sequence[T]) -> T:
    if not items:
        raise ValueError("items must not be empty")
    return items[-1]


print(last([10, 20, 30]))
print(last(["red", "green", "blue"]))
print("type parameters:", last.__type_params__)

30
blue
type parameters: (T,)


A generic relationship is now visible directly in the function definition.

Python 3.12 also introduces the `type` statement for type aliases.

In [11]:
type Point[T: int | float] = tuple[T, T]
type AdjacencyList[NodeT: Hashable] = dict[NodeT, set[NodeT]]

integer_point: Point[int] = (3, 4)
floating_point: Point[float] = (3.5, 4.25)

print(integer_point)
print(floating_point)

(3, 4)
(3.5, 4.25)


## Advanced Problem 2 — Build a generic directed graph

We will implement a directed graph with:

- generic hashable node values;
- edge insertion;
- neighbor lookup;
- breadth-first traversal;
- shortest unweighted path;
- a mapping operation that changes the node type.

The graph must preserve insertion order during traversal.

We will build it gradually.

### Step 1 — Choose the internal representation

An adjacency list is a natural representation:

```python
dict[node, set_of_neighbors]
```

However, a plain set does not preserve a useful deterministic order for tutorial output.

Instead, we will store neighbors in dictionaries whose values are `None`.

Dictionary keys preserve insertion order.

Our representation becomes:

```python
dict[NodeT, dict[NodeT, None]]
```

In [12]:
type OrderedAdjacency[NodeT: Hashable] = dict[NodeT, dict[NodeT, None]]

### Step 2 — Add nodes and edges

Adding an edge from `A` to `B` should also make sure that `B` exists in the graph, even if it has no outgoing edges.

In [13]:
class DirectedGraph[NodeT: Hashable]:
    def __init__(self) -> None:
        self._adjacency: OrderedAdjacency[NodeT] = {}

    def add_node(self, node: NodeT) -> None:
        self._adjacency.setdefault(node, {})

    def add_edge(self, source: NodeT, target: NodeT) -> None:
        self.add_node(source)
        self.add_node(target)
        self._adjacency[source].setdefault(target, None)

    def neighbors(self, node: NodeT) -> tuple[NodeT, ...]:
        if node not in self._adjacency:
            raise KeyError(node)
        return tuple(self._adjacency[node])

    def __contains__(self, node: object) -> bool:
        return node in self._adjacency

    def __len__(self) -> int:
        return len(self._adjacency)

Let's verify the basic representation before adding traversal.

In [14]:
graph = DirectedGraph[str]()
graph.add_edge("compile", "test")
graph.add_edge("compile", "lint")
graph.add_edge("test", "package")

check_equal(graph.neighbors("compile"), ("test", "lint"))
check_equal(graph.neighbors("package"), ())
check_equal(len(graph), 4)
check_equal("lint" in graph, True)
check_raises(KeyError, graph.neighbors, "missing")

KeyError('missing')

### Step 3 — Breadth-first traversal

Breadth-first search uses a queue.

We must also track visited nodes so that cycles do not cause infinite traversal.

In [15]:
def breadth_first[NodeT: Hashable](
    graph: DirectedGraph[NodeT],
    start: NodeT,
) -> Iterator[NodeT]:
    if start not in graph:
        raise KeyError(start)

    queue: deque[NodeT] = deque([start])
    visited: set[NodeT] = {start}

    while queue:
        node = queue.popleft()
        yield node

        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)


check_equal(
    list(breadth_first(graph, "compile")),
    ["compile", "test", "lint", "package"],
)

### Step 4 — Shortest unweighted path

Breadth-first search discovers nodes in increasing path length.

We can store each node's predecessor and reconstruct the path after finding the target.

In [16]:
def shortest_path[NodeT: Hashable](
    graph: DirectedGraph[NodeT],
    start: NodeT,
    target: NodeT,
) -> list[NodeT] | None:
    if start not in graph:
        raise KeyError(start)
    if target not in graph:
        raise KeyError(target)

    queue: deque[NodeT] = deque([start])
    predecessor: dict[NodeT, NodeT | None] = {start: None}

    while queue:
        node = queue.popleft()

        if node == target:
            path: list[NodeT] = []
            current: NodeT | None = target

            while current is not None:
                path.append(current)
                current = predecessor[current]

            path.reverse()
            return path

        for neighbor in graph.neighbors(node):
            if neighbor not in predecessor:
                predecessor[neighbor] = node
                queue.append(neighbor)

    return None


check_equal(
    shortest_path(graph, "compile", "package"),
    ["compile", "test", "package"],
)
check_equal(shortest_path(graph, "lint", "package"), None)

### Step 5 — Map the graph to a different node type

A generic graph should be able to transform its nodes.

For example:

```python
"compile" -> 7
"test"    -> 4
```

A mapping may cause two original nodes to become the same new node.

That is acceptable: the resulting graph naturally merges them.

### Complete Solution 2

In [17]:
class DirectedGraph[NodeT: Hashable]:
    def __init__(self) -> None:
        self._adjacency: OrderedAdjacency[NodeT] = {}

    def add_node(self, node: NodeT) -> None:
        self._adjacency.setdefault(node, {})

    def add_edge(self, source: NodeT, target: NodeT) -> None:
        self.add_node(source)
        self.add_node(target)
        self._adjacency[source].setdefault(target, None)

    def nodes(self) -> tuple[NodeT, ...]:
        return tuple(self._adjacency)

    def edges(self) -> tuple[tuple[NodeT, NodeT], ...]:
        return tuple(
            (source, target)
            for source, targets in self._adjacency.items()
            for target in targets
        )

    def neighbors(self, node: NodeT) -> tuple[NodeT, ...]:
        if node not in self._adjacency:
            raise KeyError(node)
        return tuple(self._adjacency[node])

    def map_nodes[NewNodeT: Hashable](
        self,
        transform: Callable[[NodeT], NewNodeT],
    ) -> DirectedGraph[NewNodeT]:
        mapped = DirectedGraph[NewNodeT]()

        for node in self.nodes():
            mapped.add_node(transform(node))

        for source, target in self.edges():
            mapped.add_edge(transform(source), transform(target))

        return mapped

    def __contains__(self, node: object) -> bool:
        return node in self._adjacency

    def __len__(self) -> int:
        return len(self._adjacency)

    def __repr__(self) -> str:
        return f"DirectedGraph(nodes={self.nodes()!r}, edges={self.edges()!r})"

In [18]:
build_graph = DirectedGraph[str]()
for source, target in [
    ("fetch", "parse"),
    ("parse", "validate"),
    ("validate", "store"),
    ("parse", "report"),
    ("report", "store"),
]:
    build_graph.add_edge(source, target)

check_equal(
    list(breadth_first(build_graph, "fetch")),
    ["fetch", "parse", "validate", "report", "store"],
)
check_equal(
    shortest_path(build_graph, "fetch", "store"),
    ["fetch", "parse", "validate", "store"],
)

length_graph = build_graph.map_nodes(len)
assert all(isinstance(node, int) for node in length_graph.nodes())
print(length_graph)

DirectedGraph(nodes=(5, 8, 6), edges=((5, 5), (5, 8), (5, 6), (8, 5), (6, 5)))


### Discussion

The new type-parameter syntax makes the generic relationships easier to see:

```python
class DirectedGraph[NodeT: Hashable]:
```

and:

```python
def map_nodes[NewNodeT: Hashable](...)
```

The runtime graph logic is ordinary Python.

The benefit appears primarily in:

- documentation;
- editor assistance;
- static analysis;
- reusable API design.

# 3. Precise keyword typing and intentional overrides

Python applications often pass many keyword arguments into configuration functions.

A broad annotation such as:

```python
def connect(**kwargs: object) -> None:
    ...
```

does not tell a type checker which keywords are valid.

PEP 692 allows `Unpack[TypedDict]` to describe an exact keyword schema.

Python 3.12 also adds `typing.override`, which marks a method that is intended to override a base-class method.

## A small `Unpack[TypedDict]` example

We will define a schema with one required field and several optional fields.

In [19]:
class ConnectionOptions(TypedDict):
    timeout: Required[float]
    retries: NotRequired[int]
    verify_tls: NotRequired[bool]


def connect(**options: Unpack[ConnectionOptions]) -> tuple[float, int, bool]:
    return (
        float(options["timeout"]),
        options.get("retries", 0),
        options.get("verify_tls", True),
    )


print(connect(timeout=2.5, retries=3))

(2.5, 3, True)


A static type checker can now identify:

- missing required keywords;
- misspelled keywords;
- wrong keyword value types;
- unsupported keywords.

Runtime validation is still necessary for data coming from untyped sources.

## Advanced Problem 3 — Typed notification backends

We will build a notification system with:

- an exact keyword schema;
- runtime validation;
- a base backend;
- two concrete backends;
- explicit `@override` markers;
- deterministic output for testing.

The backends will not contact real services. They will build normalized delivery records.

### Step 1 — Define the keyword schema

Each notification requires:

- `recipient`;
- `subject`.

Optional fields:

- `priority`;
- `tags`;
- `dry_run`.

A `TypedDict` describes these names precisely.

In [20]:
class NotificationOptions(TypedDict):
    recipient: Required[str]
    subject: Required[str]
    priority: NotRequired[int]
    tags: NotRequired[Sequence[str]]
    dry_run: NotRequired[bool]

### Step 2 — Normalize and validate options

We should not let each backend implement slightly different validation rules.

A shared function will:

- strip strings;
- reject empty recipient and subject;
- require priority from 1 through 5;
- deduplicate and sort tags;
- apply defaults.

In [21]:
type NormalizedNotification = tuple[
    str,
    str,
    int,
    tuple[str, ...],
    bool,
]


def normalize_notification_options(
    **options: Unpack[NotificationOptions],
) -> NormalizedNotification:
    recipient = options["recipient"].strip()
    subject = options["subject"].strip()
    priority = options.get("priority", 3)
    dry_run = options.get("dry_run", False)

    if not recipient:
        raise ValueError("recipient must not be blank")
    if not subject:
        raise ValueError("subject must not be blank")
    if not 1 <= priority <= 5:
        raise ValueError("priority must be between 1 and 5")

    tags = tuple(
        sorted(
            {
                tag.strip().casefold()
                for tag in options.get("tags", ())
                if tag.strip()
            }
        )
    )

    return recipient, subject, priority, tags, dry_run


print(
    normalize_notification_options(
        recipient=" ops@example.test ",
        subject=" Build failed ",
        priority=2,
        tags=["CI", "urgent", "ci"],
    )
)

('ops@example.test', 'Build failed', 2, ('ci', 'urgent'), False)


### Step 3 — Define the backend interface

The base class declares the method shape.

Subclasses use `@override` to document that their `deliver` methods are intended to replace the base implementation.

In [22]:
class NotificationBackend:
    def deliver(
        self,
        message: str,
        **options: Unpack[NotificationOptions],
    ) -> Mapping[str, object]:
        raise NotImplementedError

### Step 4 — Implement two backends

The email backend will produce fields resembling an email delivery.

The webhook backend will produce fields resembling a JSON webhook request.

Again, no real external communication occurs.

### Complete Solution 3

In [23]:
class EmailBackend(NotificationBackend):
    @override
    def deliver(
        self,
        message: str,
        **options: Unpack[NotificationOptions],
    ) -> Mapping[str, object]:
        recipient, subject, priority, tags, dry_run = (
            normalize_notification_options(**options)
        )

        return {
            "backend": "email",
            "to": recipient,
            "subject": subject,
            "headers": {
                "X-Priority": str(priority),
                "X-Tags": ",".join(tags),
            },
            "body": message,
            "dry_run": dry_run,
        }


class WebhookBackend(NotificationBackend):
    @override
    def deliver(
        self,
        message: str,
        **options: Unpack[NotificationOptions],
    ) -> Mapping[str, object]:
        recipient, subject, priority, tags, dry_run = (
            normalize_notification_options(**options)
        )

        return {
            "backend": "webhook",
            "endpoint": recipient,
            "payload": {
                "subject": subject,
                "message": message,
                "priority": priority,
                "tags": tags,
            },
            "dry_run": dry_run,
        }


email_delivery = EmailBackend().deliver(
    "The nightly build failed.",
    recipient="alerts@example.test",
    subject="Nightly build",
    priority=1,
    tags=["build", "nightly", "build"],
    dry_run=True,
)

webhook_delivery = WebhookBackend().deliver(
    "The nightly build failed.",
    recipient="https://hooks.example.test/build",
    subject="Nightly build",
    tags=["build"],
)

print(email_delivery)
print(webhook_delivery)

{'backend': 'email', 'to': 'alerts@example.test', 'subject': 'Nightly build', 'headers': {'X-Priority': '1', 'X-Tags': 'build,nightly'}, 'body': 'The nightly build failed.', 'dry_run': True}
{'backend': 'webhook', 'endpoint': 'https://hooks.example.test/build', 'payload': {'subject': 'Nightly build', 'message': 'The nightly build failed.', 'priority': 3, 'tags': ('build',)}, 'dry_run': False}


### Step 5 — Test the contract

At runtime, `@override` places an `__override__` marker on the method when possible.

A static type checker performs the more important validation: it verifies that the method actually overrides a compatible base method.

In [24]:
check_equal(email_delivery["backend"], "email")
check_equal(email_delivery["headers"]["X-Tags"], "build,nightly")  # type: ignore[index]
check_equal(webhook_delivery["backend"], "webhook")

check_equal(getattr(EmailBackend.deliver, "__override__", False), True)
check_equal(getattr(WebhookBackend.deliver, "__override__", False), True)

check_raises(
    ValueError,
    EmailBackend().deliver,
    "hello",
    recipient=" ",
    subject="test",
)
check_raises(
    ValueError,
    EmailBackend().deliver,
    "hello",
    recipient="x@example.test",
    subject="test",
    priority=10,
)

ValueError('priority must be between 1 and 5')

### Static-checking examples

A type checker should reject calls such as:

```python
EmailBackend().deliver(
    "hello",
    recipient="x@example.test",
)  # missing subject
```

and:

```python
EmailBackend().deliver(
    "hello",
    recipient="x@example.test",
    subject="test",
    urgency=1,
)  # unsupported keyword
```

The notebook does not execute these invalid examples because they are intended for a static checker.

# 4. PEP 709 — Inlined comprehensions

Comprehensions already looked compact before Python 3.12.

What changed is their execution model.

Python 3.12 inlines list, set, and dictionary comprehensions instead of compiling them as separate nested function-like code objects.

This generally reduces overhead and changes some tracing and introspection behavior.

## Looking for the old `<listcomp>` code object

The following function contains a list comprehension.

In [25]:
def cube_positive(values: Iterable[int]) -> list[int]:
    return [value ** 3 for value in values if value > 0]


nested_code_names = [
    constant.co_name
    for constant in cube_positive.__code__.co_consts
    if inspect.iscode(constant)
]

print("nested code objects:", nested_code_names)
print(cube_positive([-2, -1, 0, 1, 2, 3]))

assert "<listcomp>" not in nested_code_names

nested code objects: []
[1, 8, 27]


The comprehension loop variable still does not leak into the enclosing scope.

Inlining is an implementation optimization, not a return to Python 2-style variable leakage.

In [26]:
item = "outside"

result = [item * 2 for item in range(4)]

print(result)
print("outer item:", item)

check_equal(item, "outside")

[0, 2, 4, 6]
outer item: outside


## Advanced Problem 4 — Normalize transaction records

Suppose transaction records arrive from a weakly validated source.

A record is accepted only when:

- `account` is a non-blank string;
- `amount` is an integer or float but not a boolean;
- `currency` is one of `USD`, `EUR`, or `GBP`;
- `status` is `"posted"`.

Accepted records should become immutable tuples:

```python
(normalized_account, amount_as_float, normalized_currency)
```

The final result should be sorted by account, then currency, then amount.

We want to use one main comprehension, but we should avoid repeating normalization work.

### Step 1 — Understand a subtle Python type issue

Booleans are subclasses of integers:

```python
isinstance(True, int) is True
```

Therefore, this check is too broad:

```python
isinstance(amount, (int, float))
```

We must explicitly exclude booleans.

In [27]:
print(isinstance(True, int))
print(isinstance(False, float))

True
False


### Step 2 — Use assignment expressions to avoid duplicate work

The assignment expression operator `:=` lets us normalize a value once and reuse it later in the comprehension.

For example:

```python
if (account := raw_account.strip().casefold())
```

The condition both assigns and checks that the normalized account is not empty.

### Step 3 — Build the comprehension

We will use several `if` clauses.

Although this is still one comprehension, each clause expresses one validation rule.

### Complete Solution 4

In [28]:
type RawTransaction = Mapping[str, object]
type NormalizedTransaction = tuple[str, float, str]


def normalize_transactions(
    records: Iterable[RawTransaction],
) -> list[NormalizedTransaction]:
    normalized = [
        (account, float(amount), currency)
        for record in records
        if isinstance((raw_account := record.get("account")), str)
        if (account := raw_account.strip().casefold())
        if isinstance((amount := record.get("amount")), (int, float))
        if not isinstance(amount, bool)
        if math.isfinite(float(amount))
        if isinstance((raw_currency := record.get("currency")), str)
        if (currency := raw_currency.strip().upper()) in {"USD", "EUR", "GBP"}
        if record.get("status") == "posted"
    ]

    return sorted(
        normalized,
        key=lambda transaction: (
            transaction[0],
            transaction[2],
            transaction[1],
        ),
    )


raw_transactions = [
    {
        "account": "  ALPHA ",
        "amount": 10,
        "currency": "usd",
        "status": "posted",
    },
    {
        "account": "beta",
        "amount": 5.5,
        "currency": "EUR",
        "status": "pending",
    },
    {
        "account": "",
        "amount": 99,
        "currency": "GBP",
        "status": "posted",
    },
    {
        "account": "alpha",
        "amount": True,
        "currency": "USD",
        "status": "posted",
    },
    {
        "account": "beta",
        "amount": 8,
        "currency": "gbp",
        "status": "posted",
    },
]

normalized_transactions = normalize_transactions(raw_transactions)
normalized_transactions

[('alpha', 10.0, 'USD'), ('beta', 8.0, 'GBP')]

### Step 4 — Test accepted and rejected records

In [29]:
check_equal(
    normalized_transactions,
    [
        ("alpha", 10.0, "USD"),
        ("beta", 8.0, "GBP"),
    ],
)

additional_records = [
    {
        "account": "gamma",
        "amount": math.inf,
        "currency": "USD",
        "status": "posted",
    },
    {
        "account": "delta",
        "amount": 1,
        "currency": "JPY",
        "status": "posted",
    },
]

check_equal(normalize_transactions(additional_records), [])

### Step 5 — Inspect the compiled function

We can confirm that the function has no separate `<listcomp>` code object.

In [30]:
normalize_nested_code_names = [
    constant.co_name
    for constant in normalize_transactions.__code__.co_consts
    if inspect.iscode(constant)
]

print(normalize_nested_code_names)
assert "<listcomp>" not in normalize_nested_code_names

['<lambda>']


### Discussion

The comprehension is compact, but it is close to the limit of what remains comfortable to read.

A good rule is:

- use a comprehension when the transformation and filtering remain declarative;
- switch to an explicit loop when you need detailed error reporting, logging, multiple outputs, or complex branching.

PEP 709 improves the runtime model. It does not mean every loop should become a comprehension.

# 5. `itertools.batched` — Lazy fixed-size batching

Python 3.12 adds `itertools.batched`.

It groups values from an iterable into tuples of size `n`.

In [31]:
print(list(batched(range(10), 4)))

[(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]


The final batch may be shorter:

```python
[(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]
```

The function is lazy, so it works naturally with generators and large streams.

## Advanced Problem 5 — Build a resumable batch stream

We want to process a stream in batches and attach metadata to each batch.

Each emitted batch should contain:

- a one-based batch number;
- the zero-based starting offset;
- the tuple of items;
- whether the batch is complete;
- a deterministic SHA-256 checkpoint token.

The checkpoint token should be based on the textual representation of:

```python
(batch_number, starting_offset, items)
```

This is only a tutorial checkpoint format, not a secure serialization format.

### Step 1 — Define the result structure

A `TypedDict` is convenient because each result has named fields.

In [32]:
class BatchEnvelope(TypedDict):
    batch_number: int
    start_offset: int
    items: tuple[object, ...]
    complete: bool
    checkpoint: str

### Step 2 — Build a deterministic checkpoint

The same logical batch should produce the same token.

We will encode a `repr(...)` string as UTF-8 and hash it.

In [33]:
def make_checkpoint(
    batch_number: int,
    start_offset: int,
    items: tuple[object, ...],
) -> str:
    payload = repr((batch_number, start_offset, items)).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


token = make_checkpoint(1, 0, ("a", "b"))
print(token)
check_equal(len(token), 64)

ac5749b815c3653aa30ee20f1b46f4c63bce9a86fdd4b8d4aa21c18f60e1c986


### Step 3 — Track offsets correctly

If the batch size is `4`, the batches begin at offsets:

```text
0, 4, 8, 12, ...
```

Even when the last batch is incomplete, its start offset still follows this sequence.

### Complete Solution 5

In [34]:
def resumable_batches[T](
    iterable: Iterable[T],
    batch_size: int,
) -> Iterator[BatchEnvelope]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    for batch_number, items in enumerate(
        batched(iterable, batch_size),
        start=1,
    ):
        start_offset = (batch_number - 1) * batch_size
        object_items: tuple[object, ...] = items

        yield {
            "batch_number": batch_number,
            "start_offset": start_offset,
            "items": object_items,
            "complete": len(items) == batch_size,
            "checkpoint": make_checkpoint(
                batch_number,
                start_offset,
                object_items,
            ),
        }


batch_stream = resumable_batches(
    (value * 10 for value in range(9)),
    4,
)

batch_envelopes = list(batch_stream)
batch_envelopes

[{'batch_number': 1,
  'start_offset': 0,
  'items': (0, 10, 20, 30),
  'complete': True,
  'checkpoint': 'e1a298f398eb653059b501dc9ea13b999cae20ed63514cc4d7e107a0b28e4224'},
 {'batch_number': 2,
  'start_offset': 4,
  'items': (40, 50, 60, 70),
  'complete': True,
  'checkpoint': '1a817c210e99b4a2c2b798c74618726d1a9b51824c4a1745000be9e6da7540f8'},
 {'batch_number': 3,
  'start_offset': 8,
  'items': (80,),
  'complete': False,
  'checkpoint': '793c003a8376be233c116867d2ff3da2c4d0c0e767c24525424ece09ce76a814'}]

### Step 4 — Verify laziness

A lazy batching function should not consume the entire input before yielding its first result.

We can observe this with a generator that records consumption.

In [35]:
consumed: list[int] = []


def observable_source() -> Iterator[int]:
    for value in range(10):
        consumed.append(value)
        yield value


envelope_iterator = resumable_batches(observable_source(), 3)

first_envelope = next(envelope_iterator)

print("first envelope:", first_envelope)
print("consumed so far:", consumed)

check_equal(consumed, [0, 1, 2])

first envelope: {'batch_number': 1, 'start_offset': 0, 'items': (0, 1, 2), 'complete': True, 'checkpoint': '8cf88ae74583b0e3757b669559f8e7d482ec4b86efbbaaba791af44646166df8'}
consumed so far: [0, 1, 2]


### Step 5 — Test normal and incomplete batches

In [36]:
check_equal(
    [envelope["start_offset"] for envelope in batch_envelopes],
    [0, 4, 8],
)
check_equal(
    [envelope["complete"] for envelope in batch_envelopes],
    [True, True, False],
)
check_equal(
    [envelope["items"] for envelope in batch_envelopes],
    [
        (0, 10, 20, 30),
        (40, 50, 60, 70),
        (80,),
    ],
)
check_equal(
    batch_envelopes[0]["checkpoint"],
    make_checkpoint(1, 0, (0, 10, 20, 30)),
)
check_raises(ValueError, lambda: list(resumable_batches([1, 2], 0)))

ValueError('batch_size must be positive')

### Extension — Enforce full batches

Python 3.12's `batched` does not include the later `strict=` argument.

We can layer strict behavior on top.

In [37]:
def strict_batches[T](
    iterable: Iterable[T],
    batch_size: int,
) -> Iterator[tuple[T, ...]]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    for items in batched(iterable, batch_size):
        if len(items) != batch_size:
            raise ValueError(
                f"incomplete final batch: expected {batch_size}, got {len(items)}"
            )
        yield items


check_equal(
    list(strict_batches(range(6), 3)),
    [(0, 1, 2), (3, 4, 5)],
)
check_raises(
    ValueError,
    lambda: list(strict_batches(range(7), 3)),
    contains="incomplete final batch",
)

ValueError('incomplete final batch: expected 3, got 1')

# 6. `pathlib.Path.walk` — Walking directory trees with `Path` objects

Python 3.12 adds `Path.walk`.

It resembles `os.walk`, but it works naturally with `pathlib.Path`.

For each directory, it yields:

```python
(directory_path, directory_names, file_names)
```

When using top-down traversal, we can mutate `directory_names` in place to prevent traversal into selected directories.

## Advanced Problem 6 — Build a stale-cache cleanup plan

We will create a tool that scans a project tree and identifies stale cache files.

A file belongs in the cleanup plan when:

- its suffix is `.cache` or `.tmp`;
- its modification time is older than a cutoff;
- it is not under an excluded directory;
- it is a regular file.

The tool must **not** delete anything.

It should return a deterministic plan containing:

- relative path;
- size;
- age in seconds.

This separation is a safety best practice: inspect first, delete later.

## Create a temporary tutorial directory tree

In [38]:
WALK_ROOT = Path(tempfile.mkdtemp(prefix="py312_path_walk_"))

tutorial_files = {
    "src/app.py": b"print('hello')\n",
    "cache/old.cache": b"old-cache-data",
    "cache/new.cache": b"new-cache-data",
    "tmp/session.tmp": b"temporary",
    "docs/notes.txt": b"notes",
    ".git/internal.tmp": b"must-not-be-seen",
    "vendor/package.cache": b"must-also-be-excluded",
}

for relative_name, content in tutorial_files.items():
    path = WALK_ROOT / relative_name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)

print(WALK_ROOT)

C:\Users\user1\AppData\Local\Temp\py312_path_walk_b5bc915e


### Step 1 — Assign deterministic modification times

Tests based on the actual wall clock can become fragile.

Instead, we choose a fixed reference timestamp and assign known modification times.

In [39]:
REFERENCE_TIME = 2_000_000_000.0
OLD_TIME = REFERENCE_TIME - 10_000
NEW_TIME = REFERENCE_TIME - 100

import os

os.utime(WALK_ROOT / "cache/old.cache", (OLD_TIME, OLD_TIME))
os.utime(WALK_ROOT / "cache/new.cache", (NEW_TIME, NEW_TIME))
os.utime(WALK_ROOT / "tmp/session.tmp", (OLD_TIME, OLD_TIME))
os.utime(WALK_ROOT / ".git/internal.tmp", (OLD_TIME, OLD_TIME))
os.utime(WALK_ROOT / "vendor/package.cache", (OLD_TIME, OLD_TIME))

### Step 2 — Define the cleanup-plan record

A frozen dataclass provides:

- named fields;
- immutable records;
- useful equality and representation behavior.

In [40]:
@dataclass(frozen=True, slots=True)
class CleanupCandidate:
    relative_path: str
    size: int
    age_seconds: float

### Step 3 — Prune excluded directories

This is the important `Path.walk` pattern:

```python
for directory, dirnames, filenames in root.walk(top_down=True):
    dirnames[:] = [
        name for name in dirnames
        if name not in excluded
    ]
```

Replacing `dirnames` with a new list is not enough.

We mutate the original list in place using slice assignment.

### Step 4 — Calculate age

The age is:

```text
reference_time - modification_time
```

A file is stale when:

```text
age >= minimum_age
```

We will reject a negative minimum age.

### Complete Solution 6

In [41]:
def build_cleanup_plan(
    root: Path,
    *,
    reference_time: float,
    minimum_age: float,
    excluded_dirs: Iterable[str] = (),
    suffixes: Iterable[str] = (".cache", ".tmp"),
) -> list[CleanupCandidate]:
    if minimum_age < 0:
        raise ValueError("minimum_age must be non-negative")

    root = Path(root)
    excluded = set(excluded_dirs)
    normalized_suffixes = {
        suffix.casefold()
        for suffix in suffixes
    }

    candidates: list[CleanupCandidate] = []

    for directory, dirnames, filenames in root.walk(top_down=True):
        dirnames[:] = [
            dirname
            for dirname in dirnames
            if dirname not in excluded
        ]

        for filename in filenames:
            path = directory / filename

            if not path.is_file():
                continue

            if path.suffix.casefold() not in normalized_suffixes:
                continue

            stat_result = path.stat()
            age_seconds = reference_time - stat_result.st_mtime

            if age_seconds >= minimum_age:
                candidates.append(
                    CleanupCandidate(
                        relative_path=path.relative_to(root).as_posix(),
                        size=stat_result.st_size,
                        age_seconds=age_seconds,
                    )
                )

    return sorted(candidates, key=lambda item: item.relative_path)


cleanup_plan = build_cleanup_plan(
    WALK_ROOT,
    reference_time=REFERENCE_TIME,
    minimum_age=1_000,
    excluded_dirs={".git", "vendor"},
)

cleanup_plan

[CleanupCandidate(relative_path='cache/old.cache', size=14, age_seconds=10000.0),
 CleanupCandidate(relative_path='tmp/session.tmp', size=9, age_seconds=10000.0)]

### Step 5 — Verify pruning and age filtering

The plan should include:

- `cache/old.cache`;
- `tmp/session.tmp`.

It should exclude:

- the recent cache file;
- all non-cache suffixes;
- anything under `.git`;
- anything under `vendor`.

In [42]:
check_equal(
    [candidate.relative_path for candidate in cleanup_plan],
    ["cache/old.cache", "tmp/session.tmp"],
)
check_equal(
    [candidate.age_seconds for candidate in cleanup_plan],
    [10_000.0, 10_000.0],
)
check_raises(
    ValueError,
    build_cleanup_plan,
    WALK_ROOT,
    reference_time=REFERENCE_TIME,
    minimum_age=-1,
)

ValueError('minimum_age must be non-negative')

### Step 6 — Render a human-readable plan

The deletion logic remains separate.

A user or calling program can review the output before taking action.

In [43]:
def render_cleanup_plan(candidates: Sequence[CleanupCandidate]) -> str:
    if not candidates:
        return "No stale cache files found."

    total_size = sum(candidate.size for candidate in candidates)

    lines = [
        "STALE CACHE CLEANUP PLAN",
        f"files={len(candidates)}",
        f"total_size={total_size} bytes",
        "",
    ]

    lines.extend(
        f"- {candidate.relative_path} "
        f"({candidate.size} bytes, age={candidate.age_seconds:.0f}s)"
        for candidate in candidates
    )

    return "\n".join(lines)


print(render_cleanup_plan(cleanup_plan))

STALE CACHE CLEANUP PLAN
files=2
total_size=23 bytes

- cache/old.cache (14 bytes, age=10000s)
- tmp/session.tmp (9 bytes, age=10000s)


### Clean up the tutorial directory

In [44]:
shutil.rmtree(WALK_ROOT)
assert not WALK_ROOT.exists()

# 7. Numerical additions — `math.sumprod` and `math.nextafter(..., steps=...)`

Python 3.12 adds `math.sumprod`.

It computes the sum of pairwise products:

```text
p[0] * q[0] + p[1] * q[1] + ...
```

It also validates that the two inputs have equal length.

In [45]:
print(math.sumprod([1, 2, 3], [10, 20, 30]))
check_equal(math.sumprod([1, 2, 3], [10, 20, 30]), 140)
check_raises(ValueError, math.sumprod, [1, 2], [10])

140


ValueError('Inputs are not the same length')

Python 3.12 also adds a `steps` argument to `math.nextafter`.

This moves several representable floating-point values toward a target.

In [46]:
start = 1.0
after_one_step = math.nextafter(start, math.inf)
after_five_steps = math.nextafter(start, math.inf, steps=5)

print(after_one_step)
print(after_five_steps)

assert start < after_one_step < after_five_steps

1.0000000000000002
1.000000000000001


## Advanced Problem 7 — Cosine similarity with boundary exploration

Cosine similarity compares two vectors:

```text
dot(a, b) / (length(a) * length(b))
```

where:

```text
dot(a, b) = sum(a[i] * b[i])
length(a) = sqrt(dot(a, a))
```

We will:

1. validate inputs;
2. use `math.sumprod`;
3. reject zero-length vectors;
4. clamp tiny floating-point overshoots into `[-1, 1]`;
5. examine the exact neighboring floats around a threshold.

### Step 1 — Materialize iterables once

The function should accept generators.

Because we need to use each vector more than once, we materialize each input into a tuple exactly once.

### Step 2 — Validate finiteness

`NaN` and infinity can silently poison downstream calculations.

We will reject non-finite components before performing the similarity calculation.

### Step 3 — Calculate dot products with `sumprod`

We need three dot products:

- `a · b`;
- `a · a`;
- `b · b`.

A length mismatch in `a · b` automatically raises `ValueError`.

### Complete Solution 7

In [47]:
def cosine_similarity(
    left: Iterable[float],
    right: Iterable[float],
) -> float:
    left_values = tuple(float(value) for value in left)
    right_values = tuple(float(value) for value in right)

    if not all(
        math.isfinite(value)
        for value in (*left_values, *right_values)
    ):
        raise ValueError("all vector components must be finite")

    numerator = math.sumprod(left_values, right_values)
    left_squared_length = math.sumprod(left_values, left_values)
    right_squared_length = math.sumprod(right_values, right_values)

    if left_squared_length == 0 or right_squared_length == 0:
        raise ValueError("cosine similarity is undefined for a zero vector")

    similarity = numerator / math.sqrt(
        left_squared_length * right_squared_length
    )

    # Floating-point rounding may produce values such as 1.0000000000000002.
    return min(1.0, max(-1.0, similarity))


check_close(cosine_similarity([1, 0], [1, 0]), 1.0)
check_close(cosine_similarity([1, 0], [0, 1]), 0.0)
check_close(cosine_similarity([1, 0], [-1, 0]), -1.0)

### Step 4 — Test edge cases

In [48]:
check_raises(
    ValueError,
    cosine_similarity,
    [1, 2],
    [1],
)
check_raises(
    ValueError,
    cosine_similarity,
    [0, 0],
    [1, 2],
    contains="zero vector",
)
check_raises(
    ValueError,
    cosine_similarity,
    [math.nan],
    [1],
    contains="finite",
)

ValueError('all vector components must be finite')

### Step 5 — Explore a threshold one representable float at a time

Suppose an application accepts similarity values at or above `0.8`.

The floating-point value immediately below `0.8` is not equal to `0.8`.

`math.nextafter` lets us examine this exact boundary.

In [49]:
threshold = 0.8
immediately_below = math.nextafter(threshold, -math.inf)
immediately_above = math.nextafter(threshold, math.inf)

print(f"below     = {immediately_below:.18f}")
print(f"threshold = {threshold:.18f}")
print(f"above     = {immediately_above:.18f}")

assert immediately_below < threshold < immediately_above
assert not (immediately_below >= threshold)
assert immediately_above >= threshold

below     = 0.799999999999999933
threshold = 0.800000000000000044
above     = 0.800000000000000155


### Discussion

`math.nextafter` is valuable for testing numerical boundary conditions.

Instead of guessing a decimal that is “very close,” we can test the exact adjacent representable float.

That is especially useful for:

- threshold comparisons;
- numerical algorithms;
- interval boundaries;
- serialization and round-trip tests.

# 8. PEP 688 — Implementing the buffer protocol in Python

The buffer protocol allows one object to expose raw memory to another object.

Common consumers include:

- `memoryview`;
- binary parsers;
- compression functions;
- hashing functions;
- I/O APIs.

Before Python 3.12, implementing the protocol generally required C-level support.

Python 3.12 allows pure Python classes to define:

```python
__buffer__(self, flags)
```

and optionally:

```python
__release_buffer__(self, buffer)
```

## Advanced Problem 8 — Build a binary message frame

Our frame format will be:

```text
+---------+---------+------------------+
| version | flags   | payload length   |
| 1 byte  | 1 byte  | 2 bytes, big-end |
+---------+---------+------------------+
| payload bytes ...                    |
+--------------------------------------+
```

We will implement a mutable `MessageFrame` that:

- stores the entire frame in one `bytearray`;
- exposes a zero-copy `memoryview`;
- reads and writes header fields;
- validates payload length;
- tracks released views.

### Step 1 — Define header positions

Using named constants avoids unexplained numeric indexes.

In [50]:
VERSION_INDEX = 0
FLAGS_INDEX = 1
LENGTH_START = 2
LENGTH_END = 4
HEADER_SIZE = 4
MAX_PAYLOAD_SIZE = 65_535

### Step 2 — Encode and decode the two-byte payload length

The length uses big-endian byte order.

In [51]:
def encode_u16(value: int) -> bytes:
    if not 0 <= value <= 65_535:
        raise ValueError("value must fit in an unsigned 16-bit integer")
    return value.to_bytes(2, byteorder="big")


def decode_u16(data: bytes | bytearray | memoryview) -> int:
    if len(data) != 2:
        raise ValueError("exactly two bytes are required")
    return int.from_bytes(data, byteorder="big")


check_equal(encode_u16(513), b"\x02\x01")
check_equal(decode_u16(b"\x02\x01"), 513)

### Step 3 — Construct the frame

The constructor will validate:

- version and flags fit in one byte;
- payload length fits in two bytes.

The internal bytearray contains header and payload contiguously.

### Step 4 — Export a buffer

`__buffer__` returns a `memoryview` of the bytearray.

Changes through that view affect the original frame because no copy is made.

### Complete Solution 8

In [52]:
class MessageFrame:
    def __init__(
        self,
        *,
        version: int,
        flags: int,
        payload: bytes | bytearray | memoryview,
    ) -> None:
        if not 0 <= version <= 255:
            raise ValueError("version must fit in one byte")
        if not 0 <= flags <= 255:
            raise ValueError("flags must fit in one byte")

        payload_bytes = bytes(payload)

        if len(payload_bytes) > MAX_PAYLOAD_SIZE:
            raise ValueError("payload is too large")

        self._storage = bytearray(HEADER_SIZE + len(payload_bytes))
        self._storage[VERSION_INDEX] = version
        self._storage[FLAGS_INDEX] = flags
        self._storage[LENGTH_START:LENGTH_END] = encode_u16(len(payload_bytes))
        self._storage[HEADER_SIZE:] = payload_bytes

        self.released_views = 0

    def __buffer__(self, flags: int) -> memoryview:
        return memoryview(self._storage)

    def __release_buffer__(self, buffer: memoryview) -> None:
        self.released_views += 1

    @property
    def version(self) -> int:
        return self._storage[VERSION_INDEX]

    @version.setter
    def version(self, value: int) -> None:
        if not 0 <= value <= 255:
            raise ValueError("version must fit in one byte")
        self._storage[VERSION_INDEX] = value

    @property
    def flags(self) -> int:
        return self._storage[FLAGS_INDEX]

    @flags.setter
    def flags(self, value: int) -> None:
        if not 0 <= value <= 255:
            raise ValueError("flags must fit in one byte")
        self._storage[FLAGS_INDEX] = value

    @property
    def payload_length(self) -> int:
        return decode_u16(
            self._storage[LENGTH_START:LENGTH_END]
        )

    def payload_view(self) -> memoryview:
        return memoryview(self)[HEADER_SIZE:]

    def to_bytes(self) -> bytes:
        return bytes(self._storage)

    def validate(self) -> None:
        actual_payload_length = len(self._storage) - HEADER_SIZE
        if self.payload_length != actual_payload_length:
            raise ValueError(
                "header payload length does not match actual payload length"
            )


frame = MessageFrame(
    version=1,
    flags=0b0000_0011,
    payload=b"HELLO",
)

print(frame.to_bytes())
print("is Buffer:", isinstance(frame, Buffer))

b'\x01\x03\x00\x05HELLO'
is Buffer: True


### Step 5 — Demonstrate zero-copy mutation

We obtain a payload view and modify one byte.

The frame's underlying storage changes immediately.

In [53]:
payload_view = frame.payload_view()

print(bytes(payload_view))
payload_view[0] = ord("Y")
print(frame.to_bytes())

check_equal(frame.to_bytes(), b"\x01\x03\x00\x05YELLO")
check_equal(frame.payload_length, 5)

payload_view.release()

b'HELLO'
b'\x01\x03\x00\x05YELLO'


### Step 6 — Corrupt the length header and detect it

Because the full exported buffer is mutable, a consumer can change the header.

The `validate` method detects inconsistency.

In [54]:
full_view = memoryview(frame)
full_view[LENGTH_START:LENGTH_END] = encode_u16(999)

check_raises(
    ValueError,
    frame.validate,
    contains="does not match",
)

full_view[LENGTH_START:LENGTH_END] = encode_u16(5)
frame.validate()
full_view.release()

assert frame.released_views >= 2
print("released views:", frame.released_views)

released views: 2


### Discussion

The buffer protocol is powerful because it can avoid copying large binary payloads.

That power introduces responsibility:

- document whether the view is writable;
- define ownership and lifetime clearly;
- validate mutable headers;
- release views when they are no longer needed;
- avoid resizing a bytearray while exported views exist.

# 9. PEP 669 — Low-impact monitoring with `sys.monitoring`

Tools such as:

- debuggers;
- profilers;
- coverage collectors;
- observability systems

need to observe Python execution.

Python 3.12 introduces `sys.monitoring`, a lower-impact monitoring API.

Monitoring can be enabled globally or for selected code objects.

## Advanced Problem 9 — Collect line coverage for one function

We will implement:

```python
collect_line_coverage(function, *args, **kwargs)
```

It should return:

```python
(result, sorted_executed_line_numbers)
```

Requirements:

- monitor only the selected function;
- collect `LINE` events;
- always remove callbacks and free the tool ID;
- remain safe when the function raises;
- avoid leaving monitoring enabled in the notebook.

### Step 1 — Claim a tool ID

Tool IDs identify independent monitoring tools.

We will try IDs 3 and 4 and use the first available one.

In [55]:
def claim_tool_id(name: str) -> int:
    for tool_id in (3, 4):
        if sys.monitoring.get_tool(tool_id) is None:
            sys.monitoring.use_tool_id(tool_id, name)
            return tool_id

    raise RuntimeError("no available monitoring tool ID among 3 and 4")

### Step 2 — Understand the line callback

For the `LINE` event, the callback receives:

- the code object;
- the line number.

We can add each line number to a set.

### Step 3 — Use local events

We do not want to trace the entire notebook.

`set_local_events` enables events only for the selected code object.

### Complete Solution 9

In [56]:
def collect_line_coverage[ResultT](
    function: Callable[..., ResultT],
    /,
    *args: object,
    **kwargs: object,
) -> tuple[ResultT, list[int]]:
    tool_id = claim_tool_id("tutorial-line-coverage")
    line_event = sys.monitoring.events.LINE
    executed_lines: set[int] = set()

    def on_line(code: object, line_number: int) -> None:
        executed_lines.add(line_number)

    try:
        sys.monitoring.register_callback(
            tool_id,
            line_event,
            on_line,
        )
        sys.monitoring.set_local_events(
            tool_id,
            function.__code__,
            line_event,
        )

        result = function(*args, **kwargs)
        return result, sorted(executed_lines)

    finally:
        sys.monitoring.set_local_events(
            tool_id,
            function.__code__,
            sys.monitoring.events.NO_EVENTS,
        )
        sys.monitoring.register_callback(
            tool_id,
            line_event,
            None,
        )
        sys.monitoring.free_tool_id(tool_id)

### Step 4 — Profile a function with several branches

In [57]:
def classify_numbers(values: Iterable[int]) -> dict[str, int]:
    counts = {
        "negative": 0,
        "zero": 0,
        "positive_even": 0,
        "positive_odd": 0,
    }

    for value in values:
        if value < 0:
            counts["negative"] += 1
        elif value == 0:
            counts["zero"] += 1
        elif value % 2 == 0:
            counts["positive_even"] += 1
        else:
            counts["positive_odd"] += 1

    return counts


classification, covered_lines = collect_line_coverage(
    classify_numbers,
    [-2, 0, 2, 3],
)

print(classification)
print("covered lines:", covered_lines)

{'negative': 1, 'zero': 1, 'positive_even': 1, 'positive_odd': 1}
covered lines: [2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 14, 15, 17, 19]


### Step 5 — Check the result and cleanup

Exact source line numbers depend on where the function appears in the notebook, so our tests focus on stable properties:

- the function result is correct;
- some lines were observed;
- the tool IDs were freed.

In [58]:
check_equal(
    classification,
    {
        "negative": 1,
        "zero": 1,
        "positive_even": 1,
        "positive_odd": 1,
    },
)
assert covered_lines
check_equal(sys.monitoring.get_tool(3), None)
check_equal(sys.monitoring.get_tool(4), None)

### Step 6 — Verify cleanup when the monitored function raises

The `finally` block must run even when the target function fails.

In [59]:
def monitored_failure() -> None:
    marker = "before failure"
    raise RuntimeError(marker)


check_raises(
    RuntimeError,
    collect_line_coverage,
    monitored_failure,
    contains="before failure",
)
check_equal(sys.monitoring.get_tool(3), None)
check_equal(sys.monitoring.get_tool(4), None)

### Discussion

Monitoring APIs are global process resources.

Reliable tooling should always:

- claim an ID;
- register callbacks;
- enable only needed events;
- disable events;
- unregister callbacks;
- free the ID.

The cleanup path is part of the core algorithm, not an optional finishing touch.

# 10. Python 3.12 migration checks

New versions can add features and remove old APIs.

A Python 3.12 migration may need to examine code for:

- imports from the removed `distutils` package;
- deprecated `datetime.utcnow()` usage;
- deprecated boolean bitwise inversion such as `~True`.

We will build a small AST-based auditor.

It will not be a full linter, but it demonstrates how Python can analyze Python source code.

## Advanced Problem 10 — Build a migration report

The auditor will return findings containing:

- line number;
- rule code;
- explanation;
- source line.

Rules:

- `PY312-DISTUTILS`;
- `PY312-UTCNOW`;
- `PY312-BOOL-INVERT`.

### Step 1 — Define a finding record

In [60]:
@dataclass(frozen=True, slots=True)
class MigrationFinding:
    line: int
    code: str
    message: str
    source_line: str

### Step 2 — Preserve source lines

The AST provides line numbers but not automatically the exact original line text.

We will split the source into lines and retrieve the matching line safely.

In [61]:
def source_line_at(lines: Sequence[str], line_number: int) -> str:
    if 1 <= line_number <= len(lines):
        return lines[line_number - 1].strip()
    return ""

### Step 3 — Visit relevant syntax nodes

We need handlers for:

- `Import`;
- `ImportFrom`;
- `Call`;
- `UnaryOp`.

### Complete Solution 10

In [62]:
class Python312AuditVisitor(ast.NodeVisitor):
    def __init__(self, source_lines: Sequence[str]) -> None:
        self.source_lines = source_lines
        self.findings: list[MigrationFinding] = []

    def add(
        self,
        node: ast.AST,
        code: str,
        message: str,
    ) -> None:
        self.findings.append(
            MigrationFinding(
                line=node.lineno,
                code=code,
                message=message,
                source_line=source_line_at(
                    self.source_lines,
                    node.lineno,
                ),
            )
        )

    @override
    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            if alias.name == "distutils" or alias.name.startswith("distutils."):
                self.add(
                    node,
                    "PY312-DISTUTILS",
                    "distutils was removed from the standard library",
                )

        self.generic_visit(node)

    @override
    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        module = node.module or ""

        if module == "distutils" or module.startswith("distutils."):
            self.add(
                node,
                "PY312-DISTUTILS",
                "distutils was removed from the standard library",
            )

        self.generic_visit(node)

    @override
    def visit_Call(self, node: ast.Call) -> None:
        if (
            isinstance(node.func, ast.Attribute)
            and node.func.attr in {"utcnow", "utcfromtimestamp"}
        ):
            self.add(
                node,
                "PY312-UTCNOW",
                (
                    f"{node.func.attr} is deprecated; "
                    "prefer a timezone-aware UTC datetime"
                ),
            )

        self.generic_visit(node)

    @override
    def visit_UnaryOp(self, node: ast.UnaryOp) -> None:
        if (
            isinstance(node.op, ast.Invert)
            and isinstance(node.operand, ast.Constant)
            and isinstance(node.operand.value, bool)
        ):
            self.add(
                node,
                "PY312-BOOL-INVERT",
                "bitwise inversion of bool is deprecated; use logical 'not'",
            )

        self.generic_visit(node)


def audit_python_312_source(source: str) -> list[MigrationFinding]:
    source_lines = source.splitlines()
    tree = ast.parse(source)

    visitor = Python312AuditVisitor(source_lines)
    visitor.visit(tree)

    return sorted(
        visitor.findings,
        key=lambda finding: (
            finding.line,
            finding.code,
            finding.message,
        ),
    )

### Step 4 — Audit legacy code

In [63]:
legacy_code = '''
import distutils
from distutils.command.build import build
from datetime import datetime

created = datetime.utcnow()
epoch = datetime.utcfromtimestamp(0)
enabled = ~True
'''

legacy_findings = audit_python_312_source(legacy_code)

for finding in legacy_findings:
    print(
        f"{finding.line}: {finding.code}: "
        f"{finding.message}\n"
        f"    {finding.source_line}"
    )

2: PY312-DISTUTILS: distutils was removed from the standard library
    import distutils
3: PY312-DISTUTILS: distutils was removed from the standard library
    from distutils.command.build import build
6: PY312-UTCNOW: utcnow is deprecated; prefer a timezone-aware UTC datetime
    created = datetime.utcnow()
7: PY312-UTCNOW: utcfromtimestamp is deprecated; prefer a timezone-aware UTC datetime
    epoch = datetime.utcfromtimestamp(0)
8: PY312-BOOL-INVERT: bitwise inversion of bool is deprecated; use logical 'not'
    enabled = ~True


### Step 5 — Test rule coverage

In [64]:
check_equal(
    [finding.code for finding in legacy_findings],
    [
        "PY312-DISTUTILS",
        "PY312-DISTUTILS",
        "PY312-UTCNOW",
        "PY312-UTCNOW",
        "PY312-BOOL-INVERT",
    ],
)
check_equal(
    [finding.line for finding in legacy_findings],
    [2, 3, 6, 7, 8],
)

modern_code = '''
from datetime import datetime, UTC

created = datetime.now(UTC)
enabled = not True
'''

check_equal(audit_python_312_source(modern_code), [])
check_raises(SyntaxError, audit_python_312_source, "def broken(:")

SyntaxError('invalid syntax', ('<unknown>', 1, 12, 'def broken(:\n', 1, 13))

### Discussion

This auditor is intentionally conservative.

For example, it flags any call whose attribute is named `utcnow`, even if it belongs to a custom class rather than `datetime`.

A production linter would need name resolution and import analysis to reduce false positives.

Even so, this exercise demonstrates a useful migration strategy:

1. parse source;
2. search for risky syntax patterns;
3. produce actionable findings;
4. let a human review them.

# 11. Capstone — A typed telemetry processing pipeline

We will combine several Python 3.12 features in one problem.

The pipeline will:

1. accept raw telemetry mappings;
2. normalize them with an inlined comprehension;
3. represent readings with a generic class;
4. group readings with `itertools.batched`;
5. calculate weighted scores with `math.sumprod`;
6. render a report using flexible f-strings.

This problem is deliberately broken into small components.

## Capstone input

Each raw reading may contain:

```python
{
    "sensor": "north-1",
    "values": [12.0, 4.5, 8.0],
    "weights": [0.5, 0.25, 0.25],
    "status": "ok",
}
```

A valid reading must satisfy:

- non-blank sensor name;
- status equal to `"ok"`;
- values and weights are sequences;
- all values and weights are finite numbers;
- values and weights have equal length;
- weights are non-negative;
- weights sum to approximately `1.0`.

## Step 1 — Define a generic reading class

The generic class can hold a computed result of any type.

For this capstone, the result will be a float.

In [65]:
@dataclass(frozen=True, slots=True)
class Reading[ResultT]:
    sensor: str
    result: ResultT
    sample_count: int

## Step 2 — Write a weighted-score function

`math.sumprod` performs both the dot product and the equal-length check.

In [66]:
def weighted_score(
    values: Iterable[float],
    weights: Iterable[float],
) -> float:
    value_items = tuple(float(value) for value in values)
    weight_items = tuple(float(weight) for weight in weights)

    if not all(
        math.isfinite(item)
        for item in (*value_items, *weight_items)
    ):
        raise ValueError("values and weights must be finite")

    if any(weight < 0 for weight in weight_items):
        raise ValueError("weights must be non-negative")

    if not math.isclose(
        math.fsum(weight_items),
        1.0,
        rel_tol=1e-12,
        abs_tol=1e-12,
    ):
        raise ValueError("weights must sum to 1.0")

    return math.sumprod(value_items, weight_items)


check_close(
    weighted_score([10, 20], [0.25, 0.75]),
    17.5,
)

## Step 3 — Normalize one raw record

The main normalization function should provide clear exceptions.

The comprehension will be used later to select successful results.

In [67]:
type RawTelemetry = Mapping[str, object]


def normalize_one_reading(
    record: RawTelemetry,
) -> Reading[float]:
    raw_sensor = record.get("sensor")

    if not isinstance(raw_sensor, str):
        raise TypeError("sensor must be a string")

    sensor = raw_sensor.strip().casefold()

    if not sensor:
        raise ValueError("sensor must not be blank")

    if record.get("status") != "ok":
        raise ValueError("status must be 'ok'")

    values = record.get("values")
    weights = record.get("weights")

    if not isinstance(values, Sequence) or isinstance(values, (str, bytes)):
        raise TypeError("values must be a non-string sequence")

    if not isinstance(weights, Sequence) or isinstance(weights, (str, bytes)):
        raise TypeError("weights must be a non-string sequence")

    score = weighted_score(values, weights)  # type: ignore[arg-type]

    return Reading(
        sensor=sensor,
        result=score,
        sample_count=len(values),
    )

## Step 4 — Decide how to handle invalid records

In many pipelines, one invalid record should not stop all processing.

We will provide a small wrapper that returns `None` for invalid records.

A production system would normally log or collect the error details rather than silently discard them.

In [68]:
def try_normalize_reading(
    record: RawTelemetry,
) -> Reading[float] | None:
    try:
        return normalize_one_reading(record)
    except (TypeError, ValueError):
        return None

## Step 5 — Normalize many records with one comprehension

We can use an assignment expression to call the wrapper once per record.

In [69]:
def normalize_readings(
    records: Iterable[RawTelemetry],
) -> list[Reading[float]]:
    return sorted(
        [
            reading
            for record in records
            if (reading := try_normalize_reading(record)) is not None
        ],
        key=lambda item: item.sensor,
    )

## Step 6 — Batch the normalized readings

The batch summary contains:

- batch number;
- sensors;
- mean weighted score;
- minimum score;
- maximum score.

In [70]:
class TelemetryBatchSummary(TypedDict):
    batch_number: int
    sensors: tuple[str, ...]
    mean_score: float
    minimum_score: float
    maximum_score: float


def summarize_reading_batches(
    readings: Iterable[Reading[float]],
    batch_size: int,
) -> list[TelemetryBatchSummary]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    summaries: list[TelemetryBatchSummary] = []

    for batch_number, group in enumerate(
        batched(readings, batch_size),
        start=1,
    ):
        scores = tuple(reading.result for reading in group)

        summaries.append(
            {
                "batch_number": batch_number,
                "sensors": tuple(reading.sensor for reading in group),
                "mean_score": math.fsum(scores) / len(scores),
                "minimum_score": min(scores),
                "maximum_score": max(scores),
            }
        )

    return summaries

## Step 7 — Render the final report

We will use:

- quote reuse inside f-strings;
- dynamic precision;
- dynamic sensor-column width.

In [71]:
def render_telemetry_report(
    readings: Sequence[Reading[float]],
    summaries: Sequence[TelemetryBatchSummary],
    *,
    precision: int = 3,
) -> str:
    if precision < 0:
        raise ValueError("precision must be non-negative")

    sensor_width = max(
        [len("sensor"), *(len(reading.sensor) for reading in readings)]
    )

    lines = [
        "TELEMETRY REPORT",
        f"readings={len(readings)}",
        f"batches={len(summaries)}",
        "",
        (
            f"{"sensor":<{sensor_width}}"
            f"  {"samples":>7}"
            f"  {"score":>12}"
        ),
    ]

    lines.extend(
        (
            f"{reading.sensor:<{sensor_width}}"
            f"  {reading.sample_count:>7}"
            f"  {reading.result:>12.{precision}f}"
        )
        for reading in readings
    )

    lines.append("")
    lines.append("BATCHES")

    lines.extend(
        (
            f"batch={summary["batch_number"]} "
            f"sensors={",".join(summary["sensors"])} "
            f"mean={summary["mean_score"]:.{precision}f} "
            f"range=[{summary["minimum_score"]:.{precision}f}, "
            f"{summary["maximum_score"]:.{precision}f}]"
        )
        for summary in summaries
    )

    return "\n".join(lines)

## Step 8 — Run the complete capstone

In [72]:
raw_telemetry: list[RawTelemetry] = [
    {
        "sensor": " North-1 ",
        "values": [12.0, 4.0, 8.0],
        "weights": [0.5, 0.25, 0.25],
        "status": "ok",
    },
    {
        "sensor": "south-2",
        "values": [10.0, 20.0],
        "weights": [0.25, 0.75],
        "status": "ok",
    },
    {
        "sensor": "west-3",
        "values": [1.0, 2.0],
        "weights": [0.1, 0.1],
        "status": "ok",
    },
    {
        "sensor": "",
        "values": [1.0],
        "weights": [1.0],
        "status": "ok",
    },
    {
        "sensor": "east-4",
        "values": [5.0],
        "weights": [1.0],
        "status": "offline",
    },
    {
        "sensor": "central-5",
        "values": [3.0, 6.0, 9.0],
        "weights": [0.2, 0.3, 0.5],
        "status": "ok",
    },
]

normalized_readings = normalize_readings(raw_telemetry)
telemetry_summaries = summarize_reading_batches(
    normalized_readings,
    batch_size=2,
)
telemetry_report = render_telemetry_report(
    normalized_readings,
    telemetry_summaries,
    precision=2,
)

print(telemetry_report)

TELEMETRY REPORT
readings=3
batches=2

sensor     samples         score
central-5        3          6.90
north-1          3          9.00
south-2          2         17.50

BATCHES
batch=1 sensors=central-5,north-1 mean=7.95 range=[6.90, 9.00]
batch=2 sensors=south-2 mean=17.50 range=[17.50, 17.50]


## Step 9 — Test the capstone

Only three records should survive normalization:

- `central-5`;
- `north-1`;
- `south-2`.

The records with invalid weights, blank sensor, and offline status should be excluded.

In [73]:
check_equal(
    [reading.sensor for reading in normalized_readings],
    ["central-5", "north-1", "south-2"],
)
check_close(normalized_readings[0].result, 6.9)
check_close(normalized_readings[1].result, 9.0)
check_close(normalized_readings[2].result, 17.5)

check_equal(len(telemetry_summaries), 2)
check_equal(
    telemetry_summaries[0]["sensors"],
    ("central-5", "north-1"),
)
check_equal(
    telemetry_summaries[1]["sensors"],
    ("south-2",),
)

assert "TELEMETRY REPORT" in telemetry_report
assert "central-5" in telemetry_report
assert "batch=2" in telemetry_report

check_raises(
    ValueError,
    summarize_reading_batches,
    normalized_readings,
    0,
)
check_raises(
    ValueError,
    render_telemetry_report,
    normalized_readings,
    telemetry_summaries,
    precision=-1,
)

ValueError('precision must be non-negative')

## Capstone discussion

The final pipeline is not one giant function.

It is a sequence of small components:

1. `weighted_score`;
2. `normalize_one_reading`;
3. `try_normalize_reading`;
4. `normalize_readings`;
5. `summarize_reading_batches`;
6. `render_telemetry_report`.

Python 3.12 features improve individual parts:

- PEP 695 makes generic relationships clearer;
- PEP 709 reduces comprehension overhead;
- `batched` simplifies streaming groups;
- `sumprod` expresses the weighted calculation directly;
- PEP 701 makes the final report easier to write.

The architecture remains more important than the individual feature.

# 12. Review questions

Try answering these without running the notebook.

1. Why must `dirnames` be mutated with `dirnames[:] = ...` during a top-down walk?
2. Why does `isinstance(True, int)` matter when validating numerical records?
3. What does `math.sumprod` do when its inputs have different lengths?
4. Why is `try/finally` essential when using `sys.monitoring`?
5. What is the difference between static validation from `Unpack[TypedDict]` and runtime validation?
6. Why can a buffer-exporting class be dangerous if consumers can mutate header bytes?
7. What problem does `math.nextafter` solve in boundary tests?
8. Why does comprehension inlining not imply that loop variables leak?
9. What is gained by marking subclass methods with `@override`?
10. Why should a cleanup tool usually build a plan before deleting files?

# 13. Additional advanced problems

The following problems are intentionally left without full solutions.

Use the completed tutorial sections as patterns.

## Problem A — Generic dependency resolver

Extend `DirectedGraph` with:

- cycle detection;
- topological ordering;
- a useful cycle error containing the cycle path.

## Problem B — Structured f-string diagnostics

Render nested validation errors into a table with:

- dynamic field widths;
- line and column numbers;
- severity markers;
- multi-line messages.

## Problem C — Streaming upload planner

Use `batched` to build upload groups that obey both:

- a maximum file count;
- a maximum total byte size.

## Problem D — Safe duplicate cleanup

Use `Path.walk` and SHA-256 hashes to create a duplicate-file deletion plan.

Never delete the first path in each duplicate group.

## Problem E — Binary protocol parser

Extend `MessageFrame` with:

- a message-type field;
- payload checksum;
- read-only exported views;
- parser validation from arbitrary buffer objects.

## Problem F — Branch coverage

Use `sys.monitoring` branch events to build a small branch-coverage collector for one function.

## Problem G — Numerical decision boundaries

Use `math.nextafter(..., steps=...)` to generate the ten representable values immediately below and above a threshold.

## Problem H — Migration autofix suggestions

Extend the AST auditor so that each finding includes a suggested replacement snippet.

# Final summary

Python 3.12 adds features at several levels:

- syntax;
- typing;
- execution performance;
- numerical utilities;
- filesystem traversal;
- binary interoperability;
- runtime instrumentation.

The best way to learn these features is not to memorize a release-note list.

Instead:

1. identify the problem a feature solves;
2. begin with the smallest example;
3. build a realistic solution in stages;
4. test assumptions and edge cases;
5. compare the new feature with older approaches;
6. use it only when it improves clarity, safety, or performance.